# RoadTwin — Pipeline Walkthrough**SIH Problem #95** — turning a real Indian road into a simulation-readydigital road model.This notebook is the engineering story, step by step. It is deliberately**thin**: every cell calls the same functions the desktop application calls,from `core/` and `vision/`. Nothing is reimplemented here, so nothing can driftout of sync with the product.## The claim we are demonstrating> A traffic engineer can go from an address to a validated, simulation-ready> road network — and a controlled traffic experiment on it — without rebuilding> the network by hand, with a provenance chain from OSM tag to result.## The pipeline```address ─► confirmed location ─► OSM baseline ─► plain XML (editable substrate)        ─► visual AI evidence ─► human validation ─► recompile        ─► OpenDRIVE + SUMO network ─► baseline vs lane closure ─► metrics```## What you need to run this| Section | Needs ||---|---|| 1, 2, 8 | nothing || 3–7, 10 | SUMO installed, `SUMO_HOME` set || 9 | network access (Overpass, tiles) |Sections 1, 2 and 8 run anywhere — they cover the georeferencing and evidencemaths, which is where the subtle bugs live.

In [ ]:
import sys, jsonfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import config as Cprint("benchmark :", C.BENCHMARK["name"])print("coords    :", C.BENCHMARK["lat"], C.BENCHMARK["lon"])print("AOI radius:", C.BENCHMARK["aoi_radius_m"], "m")print("seeds     :", C.SIM["seeds"])print("period    :", C.SIM["period"], " <- the number that decides whether the demo works")

---## 1. Location and area of interestNothing downstream runs until a human confirms the location. That gate is thefirst thing a judge sees and the first thing that makes the tool credible: thesystem does not guess where you meant.The AOI is a square window around the confirmed point. Keep it small — buildtime scales with area, and a 500 m radius is plenty for a corridor study.

In [ ]:
from core.acquire.overpass import bbox_from_pointbbox = bbox_from_point(C.BENCHMARK["lat"], C.BENCHMARK["lon"], C.BENCHMARK["aoi_radius_m"])south, west, north, east = bboxprint(f"south {south:.6f}   west {west:.6f}")print(f"north {north:.6f}   east {east:.6f}")print(f"\nspan  {(north-south)*111320:.0f} m N-S,  {(east-west)*111320*0.975:.0f} m E-W")

---## 2. Georeferencing — the step that makes AI evidence usable at allThis is worth understanding before anything else, because it is the thing mostlikely to be got wrong silently.SAM returns a mask in **pixels**. Grounding DINO returns boxes in **pixels**.A GeoJSON file needs **degrees**. If the image the models look at is aphotograph or a screenshot, no function from pixels to degrees exists — and theentire fusion step has nothing to fuse.The fix is to build the image ourselves from XYZ map tiles. Then we know theexact bounding box of the mosaic, and pixel → lon/lat is a simple affinetransform we can prove correct.

In [ ]:
from vision.tiles import plan_mosaic, ground_resolution, deg2numzoom = C.IMAGERY["zoom"]mosaic = plan_mosaic(bbox, zoom)print(f"zoom            {mosaic.zoom}")print(f"tiles           x {mosaic.x0}..{mosaic.x1}   y {mosaic.y0}..{mosaic.y1}")print(f"mosaic size     {mosaic.width_px} x {mosaic.height_px} px")print(f"ground res      {mosaic.meters_per_pixel():.4f} m/px")print(f"bbox            {[round(v,6) for v in [mosaic.west, mosaic.south, mosaic.east, mosaic.north]]}")print()print("A 3.5 m lane is about",      round(3.5 / mosaic.meters_per_pixel()), "pixels wide at this zoom.")print("That is why lane counting from overhead imagery is feasible here.")

In [ ]:
# Prove the transform round-trips. If this is wrong, every AI coordinate is# wrong, and nothing downstream will tell you.import mathworst = 0.0for fx in (0.05, 0.5, 0.95):    for fy in (0.05, 0.5, 0.95):        px, py = fx * mosaic.width_px, fy * mosaic.height_px        lon, lat = mosaic.pixel_to_lonlat(px, py)        bx, by = mosaic.lonlat_to_pixel(lon, lat)        worst = max(worst, math.hypot(bx - px, by - py))        print(f"  ({px:7.1f},{py:7.1f}) -> ({lon:.7f},{lat:.7f}) -> ({bx:7.1f},{by:7.1f})")print(f"\nworst round-trip error: {worst:.2e} px")assert worst < 0.01print("PASS - the mosaic is genuinely georeferenced")

---## 3. OSM acquisition — the deterministic baselineWe query Overpass directly with `requests` rather than going through OSMnx.Two reasons, both practical:1. **Packaging.** OSMnx pulls in GeoPandas and NetworkX, which are among the   hardest libraries to freeze with PyInstaller. Our installer has to work.2. **Format.** Overpass hands back raw `.osm` XML, which is exactly what   `netconvert` wants as input. OSMnx would hand back a graph object we would   have to convert.The extract is cached on disk and committed to the repository. That is not anoptimisation — Overpass rate-limits and venue wifi fails, and this file is thedifference between a demo and an apology.

In [ ]:
from core.acquire.overpass import fetch_osmproj = C.PROJECTS_DIR / "notebook"osm = fetch_osm(    bbox, proj / "build" / "benchmark.osm",    endpoint=C.OVERPASS["endpoint"],    user_agent=C.OVERPASS["user_agent"],    cache_dir=C.BENCHMARK_DIR,    allow_network=True,          # set False to force the cached copy)print(f"\n{osm}  ({osm.stat().st_size/1024:.0f} KB)")

---## 4. Compilation — and the decision that saved the project**We do not write an OpenDRIVE compiler.**OpenDRIVE is a geometry specification, not a serialisation format. Producing a`.xodr` that imports into a *connected, drivable* network requires referencelines with continuous arc-length parameters, lane sections with correct signconventions, predecessor/successor links on every road and every lane, andjunctions expressed as connecting roads with consistent lane links.The failure mode is not a crash. `netconvert` imports the file, reports little,and produces a network where every junction is disconnected — so no vehicle canroute, and the metrics come back empty. That is a day spent diagnosing, notwriting.`netconvert` already imports OSM **and** exports OpenDRIVE. So we use it forboth directions, and the compiler we would have written disappears.```benchmark.osm ─[netconvert]─► plain XML (.nod .edg .con .tll)   ◄── EDITS LAND HERE                                    │                              [netconvert]                              ╱           ╲                network.net.xml           road_network.xodr```The plain XML files are small, readable, and round-trip losslessly. They arethe **single editable substrate**: change `numLanes` on an edge, recompile, andboth the SUMO network and the OpenDRIVE follow.

In [ ]:
from core.build.netconvert import osm_to_plain, plain_to_net, verify_xodr_roundtripplain = osm_to_plain(osm, proj / "build")for k, v in plain.items():    if v.exists():        print(f"  {k:5} {v.name:24} {v.stat().st_size/1024:8.1f} KB")

In [ ]:
# Look at the substrate. This is the whole model, and it is human-readable.print(plain["edg"].read_text()[:1200])

In [ ]:
out = plain_to_net(plain, proj / "sumo" / "network.net.xml",                   xodr_out=proj / "road_network.xodr")print(f"network : {out['net'].stat().st_size/1024:.0f} KB")print(f"xodr    : {out['xodr'].stat().st_size/1024:.0f} KB")# "Is your OpenDRIVE actually valid?" -- answer with evidence, not assertion.ok = verify_xodr_roundtrip(out["xodr"], proj / "build" / "roundtrip.net.xml")print(f"\nOpenDRIVE re-imports cleanly: {'PASS' if ok else 'FAIL'}")

In [ ]:
print(out["xodr"].read_text()[:900])

---## 5. Inspecting the networkBefore simulating anything, look at what we built. A network that looks wronghere will look wrong for five days.

In [ ]:
from core.sim.scenario import read_net_edges, pick_closure_candidateedges = read_net_edges(out["net"])print(f"{len(edges)} drivable edges, {sum(e['num_lanes'] for e in edges.values())} lanes\n")rows = sorted(edges.items(), key=lambda kv: -kv[1]["length_m"])[:10]print(f"{'edge':<24}{'lanes':>6}{'length m':>10}{'speed kph':>11}")print("-" * 51)for eid, d in rows:    kph = d["lanes"][0]["speed"] * 3.6 if d["lanes"] else 0    print(f"{eid:<24}{d['num_lanes']:>6}{d['length_m']:>10.1f}{kph:>11.0f}")cand = pick_closure_candidate(out["net"])print(f"\nclosure candidate: {cand}")

---## 6. Demand — the number that decides whether the demo worksA lane closure on an empty road changes nothing.If the experiment reports "42 s baseline, 42 s closure", the whole projectlooks pointless — and that is a **demand calibration failure**, not a modellingfailure. The corridor has to be near saturation for removing a lane to matter.Target roughly 75–85% of capacity, then stop. Push further and vehicles startteleporting (SUMO's gridlock escape hatch), at which point the metrics describea broken simulation rather than congestion.

In [ ]:
from core.sim.demand import generate_routesd = generate_routes(    out["net"], proj / "sumo",    begin=C.SIM["begin"], end=C.SIM["end"],    period=C.SIM["period"], fringe_factor=C.SIM["fringe_factor"],    seed=C.SIM["seeds"][0],)print(f"\n{d['count']} vehicles over {C.SIM['end']}s "      f"= {d['count']/(C.SIM['end']/3600):.0f} veh/h offered")

---## 7. The experiment — a controlled comparison, not a demo trickThe closure is a **rerouter additional-file**, not a network edit.That distinction is the difference between a result and an anecdote. If werebuilt the network to remove a lane, edge ids and routes would change, and thetwo runs would no longer be comparable. With a rerouter, both runs load theidentical network, the identical routes and the identical seeds — so the onlyvariable is the closure, and the measured difference is causal.**One honest limitation:** SUMO closes a lane for the whole length of an edge.There is no partial-length lane closure. So we report the *actual* closedlength rather than claiming a round number we did not model.

In [ ]:
from core.sim.scenario import build_lane_closuredesc = build_lane_closure(    out["net"], proj / "sumo" / "closure.add.xml",    edge_id=cand["edge_id"],    lane_index=cand["num_lanes"] - 1,     # leftmost lane    begin=C.CLOSURE["begin"], end=C.CLOSURE["end"],)print(json.dumps(desc, indent=2))print()print((proj / "sumo" / "closure.add.xml").read_text())

In [ ]:
from core.sim.run import run_experimentresult = run_experiment(    out["net"], d["routes"], proj / "sumo" / "results",    proj / "sumo" / "closure.add.xml",    seeds=C.SIM["seeds"][:3],           # 3 seeds in the notebook; 5 for the demo    begin=C.SIM["begin"], end=C.SIM["end"],    closed_edge=cand["edge_id"],)print()print(result["table"])

### Reading the result honestly`comparison.significant` is `False` when the change is no larger thanseed-to-seed variation. **A non-significant result is not a finding.** If yousee it, lower `SIM["period"]` in `config.py` and re-run — do not present noiseas evidence.Reporting a mean over several seeds costs about twenty seconds of runtime andconverts the number from an anecdote into a measurement.

In [ ]:
cmp = result["comparison"]print("significant:", cmp["significant"])print("verdict    :", cmp["verdict"])print()for r in cmp["rows"]:    print(f"  {r['metric']:<22} {r['baseline']} -> {r['scenario']}  ({r['delta_pct']:+}%)")

---## 8. Visual evidence — from a mask to an engineering claimThe interesting AI output here is not "we found a traffic light". It is a**measurement**:> The road surface is 10.8 m wide. At a nominal 3.5 m per lane that is 3 lanes.> OSM says 2. Confidence 82%.That is a claim an engineer can act on, and it is exactly the disagreement thereview interface exists to resolve.The method: sample points along the OSM centerline, measure the mask widthalong the perpendicular at each, convert pixels to metres using the mosaic'sground resolution, divide by the nominal lane width.The cells below use a **synthetic mask**, so they run without a GPU or networkand demonstrate the maths directly.

In [ ]:
import numpy as npfrom vision.evidence import lane_count_evidencempp = 0.25                       # metres per pixel, roughly zoom 19H, W = 400, 800def synthetic_road(width_m, jitter_px=0, seed=0):    rng = np.random.default_rng(seed)    mask = np.zeros((H, W), dtype=bool)    for x in range(W):        half = int(round((width_m / mpp) / 2)) + (int(rng.integers(-jitter_px, jitter_px+1)) if jitter_px else 0)        mask[max(H//2 - half, 0): min(H//2 + half, H), x] = True    centre = [(float(x), float(H//2)) for x in range(20, W-20, 20)]    return mask, centrefor width in (7.0, 10.5, 14.0):    mask, centre = synthetic_road(width)    ev = lane_count_evidence(mask, centre, mpp)    print(f"true {width:5.1f} m  ->  measured {ev['measured_width_m']:5.2f} m  "          f"->  {ev['value']} lanes   confidence {ev['confidence']:.2f}")

In [ ]:
# Confidence has to react to reality, or it is decoration.print("clean, unambiguous road surface:")mask, centre = synthetic_road(10.5)print("  ", lane_count_evidence(mask, centre, mpp)["confidence"])print("\nragged mask edges (width varies along the road):")mask, centre = synthetic_road(10.5, jitter_px=9)print("  ", lane_count_evidence(mask, centre, mpp)["confidence"])print("\nwidth halfway between 3 and 4 lanes -- exactly when a human should look:")mask, centre = synthetic_road(12.25)ev = lane_count_evidence(mask, centre, mpp)print(f"   {ev['confidence']}   (raw estimate {ev['raw_estimate']} lanes)")print("\nnot enough usable samples:")mask, centre = synthetic_road(10.5)print("  ", lane_count_evidence(mask, centre[:2], mpp), " <- refuses to guess")

Refusing to answer is a feature. A confident lane count derived from threenoisy samples is how a demo gets taken apart in questioning.### On the real imageryTwo things make the real pipeline work, both learned the hard way:**Use Hugging Face `transformers`, not the upstream repos.** TheIDEA-Research Grounding DINO repo compiles a custom CUDA operator at installtime and fails constantly on Windows. The `transformers` ports are purePyTorch.**Prompt SAM with the OSM centerline.** Do not ask SAM to segment everythingand then guess which blob is the road — you already know where the road is.Feeding centerline points as positive prompts turns a generic segmenter into aroad-surface extractor.```pythonfrom vision.segment import segment_roadfrom vision.evidence import lane_count_evidence, make_observationcenterline_px = [mosaic.lonlat_to_pixel(lon, lat) for lon, lat in road_geometry]mask = segment_road("mosaic.png", centerline_px)ev   = lane_count_evidence(mask, centerline_px, mosaic.meters_per_pixel())obs  = make_observation("obs-001", ev, road_id="rt-road-007")```

---## 9. Fusion and human validationThe design commitment that shapes the whole product:> **AI produces evidence. The engineer decides. Every decision is recorded.**An AI road builder that is wrong 15% of the time is unusable. An AI evidencegenerator that is right 85% of the time, with a human gate, is valuable. Somodel output never mutates the model — it creates a review item.Observations are immutable. Rejecting one sets its status; it is never deleted.

In [ ]:
from vision.evidence import build_review_itemsobservations = [{    "id": "obs-001", "source": "sam2", "feature": "lane_count", "value": 3,    "confidence": 0.82, "geometry_kind": "georeferenced",    "attached_to": {"road_id": "rt-road-007"},}]# Note the baseline: OSM had NO lanes tag, so the value was inferred.# Saying so is more honest than pretending OSM supplied it -- and it is a# better demo, because incomplete map data is the real problem here.baseline = {"rt-road-007": {    "lane_count": 2,    "lane_count_provenance": {"source": "inferred_default",                              "rule": "highway=trunk -> 2 lanes",                              "confidence": 0.4},}}for item in build_review_items(observations, baseline):    print(json.dumps(item, indent=2))

Which the interface presents as:```ROAD FEATURE — lane countOSM:      no data   (inferred: 2, confidence 40%)VISION:   3 lanes   (mask width 10.8 m / 3.5 m, confidence 82%)[ ACCEPT VISION ]   [ KEEP BASELINE ]   [ EDIT ]```

---## 10. Accept → recompile → re-simulateAccepting a review item produces an `Edit`. The edit rewrites the plain`.edg.xml`; one `netconvert` call regenerates the SUMO network **and** theOpenDRIVE; the experiment re-runs.One funnel, one source of truth, no divergence — and an audit trail thatsatisfies the invariant:> `baseline plain XML + validation_report.json == final model`That is tested, not asserted.

In [ ]:
from core.model.edits import Edit, apply_edits, read_edges, replay, write_validation_reporttarget = cand["edge_id"]before = read_edges(plain["edg"])[target].get("numLanes")edit = Edit(edge_id=target, attribute="numLanes", old_value=None, new_value="3",            source="accepted_vision", observation_id="obs-001", confidence=0.82,            user_action="accept")apply_edits(plain["edg"], [edit], proj / "build" / "plain_edited.edg.xml")after = read_edges(proj / "build" / "plain_edited.edg.xml")[target].get("numLanes")print(f"{target}: numLanes {before} -> {after}")report = write_validation_report([edit], observations, proj / "validation_report.json")print(f"audit trail written: {report.name}")

In [ ]:
# The replayability claim, demonstrated rather than asserted.replayed = replay(plain["edg"], report, proj / "build" / "plain_replayed.edg.xml")value = read_edges(replayed)[target].get("numLanes")print(f"replayed from baseline + report -> numLanes = {value}")assert value == "3"print("PASS - the final model is reproducible from the baseline and the decisions alone")

In [ ]:
# Recompile from the edited substrate: new network AND new OpenDRIVE.edited = dict(plain); edited["edg"] = proj / "build" / "plain_edited.edg.xml"out2 = plain_to_net(edited, proj / "sumo" / "network_edited.net.xml",                    xodr_out=proj / "road_network_edited.xodr")e_before = read_net_edges(out["net"])[target]["num_lanes"]e_after  = read_net_edges(out2["net"])[target]["num_lanes"]print(f"\nSUMO network lanes on {target}: {e_before} -> {e_after}")print("The OpenDRIVE export was regenerated in the same call.")

---## 11. ExportEverything, with provenance, in one zip a judge can take away.

In [ ]:
from core.export.package import export_project, write_readmewrite_readme(proj, location={"name": C.BENCHMARK["name"], "lat": C.BENCHMARK["lat"],                             "lon": C.BENCHMARK["lon"],                             "aoi_radius_m": C.BENCHMARK["aoi_radius_m"]},             results_table=result["table"])zp = export_project(proj, C.PROJECTS_DIR / "RoadTwin_Project_notebook.zip")import zipfilefor n in sorted(zipfile.ZipFile(zp).namelist()):    print("  ", n)

---## What this walkthrough established| Step | Claim | How it was shown ||---|---|---|| 2 | AI output can be georeferenced | pixel↔lon/lat round-trip to <0.01 px || 4 | The OpenDRIVE is valid | re-imported through netconvert || 5 | The network is real | edges, lanes, lengths, speeds listed || 7 | The experiment is controlled | same net, same routes, same seeds || 7 | The result is not noise | mean over seeds + significance check || 8 | The AI claim is quantitative | width measurement → lane count + confidence || 8 | Confidence is meaningful | drops on ragged and ambiguous inputs || 9 | AI does not decide | review items, immutable observations || 10 | The trail is auditable | replayed the final model from the report |### The four questions to expect**"Isn't this just netconvert?"** — netconvert is our compiler backend, the wayLLVM is a compiler backend. The contribution is the layer above it:georeferenced evidence, a human validation gate, and a provenance chain.Rewriting a mature compiler would have been the least novel and most fragilepart of the system.**"How do you know the AI is right?"** — We don't, and the architecture assumeswe don't. That is why nothing is committed silently.**"Why does a lane closure matter?"** — It is the smallest experiment thatproves the model is simulation-grade rather than a picture.**"What was genuinely hard?"** — Georeferencing visual evidence so it can becompared against map data at all, and keeping an auditable chain from OSM tagto simulation result.